In [1]:
import json
import glob
from pathlib import Path, WindowsPath
import pathlib

import pandas as pd
import numpy as np

import numba

import csv
import threading

import re

In [2]:
pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
df = pd.DataFrame()

In [3]:
# Ordering functions
def ordering_function_tf(path):
    """Give the timeframe contained in the name of the json file as an integer.
    Use to sort metadata files"""
    
    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(1))

def ordering_function_cell(path):
    """Give the cell number contained in the name of the json file as an integer.
    Use to sort metadata files"""

    pattern = r'mask_tf(\d+)_apoc_cell(\d+)\.json'
    match = re.match(pattern, path.name)
    return int(match.group(2))

In [4]:
def process_subdirectory(subdirectory_path, pattern=pattern, dic={}):
    json_files = list(subdirectory_path.glob('mask_tf*_apoc_cell*.json'))
    json_files.sort(key=lambda x: (ordering_function_cell(x), ordering_function_tf(x)))
    cells = {re.match(pattern, file.name).group(2) for file in json_files}  # Cell numbers as in the metadata file names

    # Check whether cells is empty or metadata has been found, if so populate df with average intensity
    if cells:
        for cell in cells:
            print(f'Recording contour of cell {subdirectory.name}_{cell}')
            column_name = subdirectory_path.name + '_' + cell
            json_cell = [json_file for json_file in json_files if 'apoc_cell' + cell in json_file.stem]
            tfs = [int((re.match(pattern, json_file.name)).group(1)) for json_file in json_cell]
            tfs.sort()

            for (tf, json_file) in zip(tfs, json_cell):
                with json_file.open('r') as file:
                    data_dict = json.load(file)
                    dic.setdefault(column_name, {}).setdefault(tf, {
                        'x_coords': data_dict.get('xcoords'),
                        'y_coords': data_dict.get('ycoords')
                    })
    else:
        print(f'No cell metadata in {subdirectory_path.name}')

    return dic

In [19]:
cd H:/PROJECTS-03/Pablo/oscillating/ppf021_analysis_th_corr/

H:\PROJECTS-03\Pablo\oscillating\ppf021_analysis_th_corr


In [6]:
root_folder = Path("./raw/metadata/ppf025")
subdirectories = list(root_folder.glob('*/'))

In [7]:
subdirectories = [subdir for subdir in subdirectories if subdir.is_dir()]

In [8]:
dic= {}
for subdirectory in subdirectories:
    process_subdirectory(subdirectory, dic=dic)

Recording contour of cell ppf025_xy001_0
Recording contour of cell ppf025_xy002_0
Recording contour of cell ppf025_xy003_0
Recording contour of cell ppf025_xy004_0
Recording contour of cell ppf025_xy005_0
Recording contour of cell ppf025_xy006_0
Recording contour of cell ppf025_xy007_1
Recording contour of cell ppf025_xy007_0
Recording contour of cell ppf025_xy008_0
Recording contour of cell ppf025_xy009_1
Recording contour of cell ppf025_xy009_0
Recording contour of cell ppf025_xy010_0
Recording contour of cell ppf025_xy011_0
Recording contour of cell ppf025_xy012_0
Recording contour of cell ppf025_xy013_0
Recording contour of cell ppf025_xy014_0
Recording contour of cell ppf025_xy015_0
Recording contour of cell ppf025_xy016_0
Recording contour of cell ppf025_xy017_0
Recording contour of cell ppf025_xy018_0
Recording contour of cell ppf025_xy019_0
Recording contour of cell ppf025_xy020_0
Recording contour of cell ppf025_xy021_1
Recording contour of cell ppf025_xy021_0
Recording contou

In [11]:
len(dic)

141

In [12]:
dic.

SyntaxError: invalid syntax (625444278.py, line 1)

In [10]:
# Initialize lists to store data for DataFrame
data = []

# Extract data from the nested dictionary
for column_name, time_data in dic.items():
    #cell_name = column_name.split('_')[-1]
    cell_name = column_name
    
    for tf, coord_data in time_data.items():
        data.append({
            'Cell Name': cell_name,
            'Time Frame': tf,
            'x_coords': coord_data.get('x_coords', []),
            'y_coords': coord_data.get('y_coords', [])
        })

# Create the DataFrame
df = pd.DataFrame(data)

# Optionally, set the time frame as the index
# df.set_index('Time Frame', inplace=True)

# Display the DataFrame
print(df)

           Cell Name  Time Frame  \
0     ppf025_xy001_0           0   
1     ppf025_xy001_0           1   
2     ppf025_xy001_0           2   
3     ppf025_xy001_0           3   
4     ppf025_xy001_0           4   
...              ...         ...   
8304  ppf025_xy101_0          70   
8305  ppf025_xy101_0          71   
8306  ppf025_xy101_0          72   
8307  ppf025_xy101_0          73   
8308  ppf025_xy101_0          74   

                                               x_coords  \
0     [253.0, 254.0, 255.0, 256.0, 257.0, 258.0, 259...   
1     [253.0, 254.0, 255.0, 256.0, 257.0, 258.0, 259...   
2     [252.0, 253.0, 254.0, 255.0, 255.5, 256.0, 257...   
3     [248.0, 249.0, 250.0, 251.0, 252.0, 253.0, 254...   
4     [253.0, 254.0, 255.0, 256.0, 257.0, 257.5, 258...   
...                                                 ...   
8304  [250.0, 251.0, 252.0, 253.0, 253.5, 254.0, 255...   
8305  [250.0, 250.5, 251.0, 252.0, 253.0, 254.0, 255...   
8306  [245.0, 246.0, 247.0, 248.0, 2

In [11]:
df.to_csv('./contour_data_ppf025_median_corr')

In [147]:
df.iloc[:,0]

Time Frame
0     ppf005_xy002_0
1     ppf005_xy002_0
2     ppf005_xy002_0
3     ppf005_xy002_0
4     ppf005_xy002_0
           ...      
75    ppf005_xy174_0
76    ppf005_xy174_0
77    ppf005_xy174_0
78    ppf005_xy174_0
79    ppf005_xy174_0
Name: Cell Name, Length: 10072, dtype: object

In [15]:
df

,Cell Name,Time Frame,x_coords,y_coords
0,ppf021_xy001_0,0,"[274.0, 275.0, 276.0, 277.0, 278.0, 279.0, 280...","[220.5, 220.5, 220.5, 220.5, 220.5, 220.5, 220..."
1,ppf021_xy001_0,1,"[273.0, 273.5, 274.0, 275.0, 276.0, 277.0, 277...","[222.5, 223.0, 223.5, 223.5, 223.5, 223.5, 223..."
2,ppf021_xy001_0,2,"[267.0, 268.0, 269.0, 270.0, 271.0, 272.0, 272...","[221.5, 221.5, 221.5, 221.5, 221.5, 221.5, 222..."
3,ppf021_xy001_0,3,"[270.0, 271.0, 272.0, 273.0, 274.0, 274.5, 275...","[226.5, 226.5, 226.5, 226.5, 226.5, 227.0, 227..."
4,ppf021_xy001_0,4,"[263.0, 263.5, 264.0, 265.0, 265.5, 266.0, 266...","[219.5, 220.0, 220.5, 220.5, 220.0, 219.5, 220..."
...,...,...,...,...
9911,ppf021_xy110_0,72,"[177.0, 178.0, 179.0, 180.0, 180.5, 181.0, 182...","[78.5, 78.5, 78.5, 78.5, 79.0, 79.5, 79.5, 79...."
9912,ppf021_xy110_0,73,"[175.0, 176.0, 177.0, 178.0, 179.0, 180.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 79...."
9913,ppf021_xy110_0,74,"[176.0, 177.0, 178.0, 179.0, 180.0, 181.0, 181...","[79.5, 79.5, 79.5, 79.5, 79.5, 79.5, 80.0, 80...."
9914,ppf021_xy110_0,75,"[178.0, 179.0, 179.5, 180.0, 181.0, 182.0, 183...","[78.5, 78.5, 79.0, 79.5, 79.5, 79.5, 79.5, 79...."


In [121]:
df.loc[df['Cell Name']=='ppf005_xy003_0', 'x_coords']

Time Frame
0     [265.0, 266.0, 267.0, 268.0, 269.0, 270.0, 271...
1     [263.0, 264.0, 265.0, 265.5, 266.0, 267.0, 267...
2     [263.0, 264.0, 265.0, 266.0, 267.0, 268.0, 268...
3     [263.0, 264.0, 265.0, 266.0, 266.5, 267.0, 268...
4     [262.0, 263.0, 264.0, 265.0, 266.0, 267.0, 267...
                            ...                        
75    [262.0, 263.0, 264.0, 265.0, 266.0, 267.0, 268...
76    [261.0, 262.0, 262.5, 263.0, 264.0, 264.5, 265...
77    [260.0, 260.5, 261.0, 261.5, 262.0, 263.0, 264...
78    [259.0, 260.0, 260.5, 261.0, 261.5, 262.0, 263...
79    [259.0, 259.5, 260.0, 260.5, 261.0, 262.0, 263...
Name: x_coords, Length: 80, dtype: object

In [ ]:
## Contour extraction with multithread
def multi_threaded_file_reader(file_paths):
    threads = []
    results = {}

    # Define the worker function
    def read_file_thread(file_path):
        result = read_file(file_path)
        results[file_path] = result

    # Create and start threads
    for file_path in file_paths:
        thread = threading.Thread(target=read_file_thread, args=(file_path,))
        threads.append(thread)
        thread.start()

    # Wait for all threads to finish
    for thread in threads:
        thread.join()

    return results

In [159]:
subdirectories_paths = list(map(lambda path: path.as_posix(), subdirectories))

In [ ]:
dic= {}
for subdirectory in subdirectories:
    process_subdirectory(subdirectory, dic=dic)
    

In [ ]:
import concurrent.futures

def process_subdirectory(subdirectory_path, pattern=pattern, dic={}):
    json_files = list(subdirectory_path.glob('mask_tf*_apoc_cell*.json'))
    json_files.sort(key=lambda x: (ordering_function_cell(x), ordering_function_tf(x)))
    cells = {re.match(pattern, file.name).group(2) for file in json_files}  # Cell numbers as in the metadata file names

    # Check whether cells is empty or metadata has been found, if so populate df with average intensity
    if cells:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = []

            for cell in cells:
                print(f'Submitting task for cell {subdirectory_path.name}_{cell}')
                column_name = subdirectory_path.name + '_' + cell
                json_cell = [json_file for json_file in json_files if 'apoc_cell' + cell in json_file.stem]
                tfs = [int((re.match(pattern, json_file.name)).group(1)) for json_file in json_cell]
                tfs.sort()

                for (tf, json_file) in zip(tfs, json_cell):
                    futures.append(executor.submit(process_json_file, tf, json_file, column_name, dic))

            # Wait for all tasks to complete
            concurrent.futures.wait(futures)

    else:
        print(f'No cell metadata in {subdirectory_path.name}')

    return dic

def process_json_file(tf, json_file, column_name, dic):
    with json_file.open('r') as file:
        data_dict = json.load(file)
        dic.setdefault(column_name, {}).setdefault(tf, {
            'x_coords': data_dict.get('xcoords'),
            'y_coords': data_dict.get('ycoords')
        })

if __name__ == "__main__":
    # Example usage:
    import pathlib

    subdirectories = [pathlib.Path("subdir1"), pathlib.Path("subdir2")]
    for subdirectory in subdirectories:
        process_subdirectory(subdirectory)